In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

inp = Path('/kaggle/input')
print('top-level:', [p.name for p in inp.iterdir()] if inp.exists() else None)

def discover_root() -> Path:
    if not inp.exists():
        raise FileNotFoundError('/kaggle/input missing')
    candidates = []
    for child in sorted(inp.iterdir()):
        candidates.append(child)
        if child.is_dir():
            for sub in sorted(child.iterdir()):
                candidates.append(sub)
                # one more level: /kaggle/input/competitions/<slug>
                if sub.is_dir():
                    for sub2 in sorted(sub.iterdir()):
                        candidates.append(sub2)
    for c in candidates:
        if c.is_dir() and (c / 'sample_submission.csv').exists():
            return c
    raise FileNotFoundError(
        'sample_submission.csv not found; scanned: '
        + str([str(c) for c in candidates[:50]])
    )

ROOT = discover_root()
print('ROOT', ROOT)
sample = pd.read_csv(ROOT / 'sample_submission.csv')
train = pd.read_csv(ROOT / 'train.csv') if (ROOT / 'train.csv').exists() else None
study_col = 'StudyInstanceUID'
targets = [c for c in sample.columns if c != study_col]
print('n_test', len(sample))

prev = {}
for t in targets:
    if train is not None and t in train.columns and train[t].notna().any():
        prev[t] = float(train[t].mean())
    else:
        prev[t] = 0.5

rng = np.random.default_rng(42)
out = sample[[study_col]].copy()
for t in targets:
    out[t] = np.clip(prev[t] + rng.normal(0.0, 0.01, size=len(sample)), 1e-6, 1 - 1e-6)
out = out[sample.columns]
assert list(out[study_col].astype(str)) == list(sample[study_col].astype(str))

out_path = Path('/kaggle/working/submission.csv')
out.to_csv(out_path, index=False)
print('Wrote', out_path, out.shape)
print(out.head())
